# Step8 — HCI Compuesto y Análisis de Sensibilidad (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 8 — Índice de Cronicidad de Inundaciones por El Niño (compuesto)
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-05

**Objetivo de este notebook:** combinar los cuatro índices ya construidos
(Severidad — Fase 4, Recurrencia — Fase 5, Persistencia — Fase 6, Magnitud
ENSO — Fase 7) en el HCI compuesto (Fase 0 §9):

```
HCI = 0.35·Recurrencia + 0.30·Severidad + 0.20·Persistencia + 0.15·Magnitud ENSO
```

y ejecutar el análisis de sensibilidad que la Fase 0 exige explícitamente
antes de fijar los pesos, más los **cuatro** parámetros específicos ya
señalados como pendientes en cada fase anterior:

1. Fase 4 §6 — dos valores atípicos que dominan el min-max de Severidad
   (Quito, 750,000 afectados indirectos; Baños de Agua Santa, 500,000
   muertes de ganado).
2. Fase 5 §10 — la definición de "evento ENSO" (mes a mes, no por
   episodios consecutivos).
3. Fase 6 §4.2 — pesos iguales (1/3) entre los tres componentes de
   Persistencia, sin fórmula original que los fije.
4. Fase 7 §6 — la elección entre `enso_magnitude_max` (baja discriminación)
   y `enso_magnitude_mean` (buena discriminación), dejada pendiente a
   propósito para esta fase.

**Clasificación final confirmada por Jerson (AskUserQuestion, 2026-08-05):**
quiebres naturales (Jenks), no cuantiles — la distribución del HCI es
sesgada (pocos cantones muy extremos), y Jenks encuentra agrupamientos
reales en vez de forzar grupos de igual tamaño.

**English:** Composite Index (HCI)

Combines recurrence, severity, persistence and ENSO magnitude into the composite chronicity index, classifies it with natural breaks (Jenks) and runs the first sensitivity analyses (ENSO magnitude version, alternative weights, robustness to outliers).

Outputs: `hci_composite_*` tables in `data/processed/`.

## 0. Configuración del entorno

**English:** 0. Setup

In [ ]:
# Fase 8 tampoco requiere GEE ni CDS.

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # notebook vive en <repo>/notebooks/
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd  # tablas
# Numerical operations
import numpy as np  # operaciones numéricas
# Efficient reader for GeoPackage layers
import pyogrio  # lectura eficiente de hazard_event_geo.gpkg
# Natural-breaks (Jenks) classification
import jenkspy  # clasificación por quiebres naturales (Jenks)
# Spearman rank correlation
from scipy.stats import spearmanr  # correlación de rangos para el análisis de sensibilidad

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step8_hci_composite", config, log_dir=PROJECT_ROOT / "logs")
logger.info("Fase 8 iniciada — configuración cargada desde %s", CONFIG_PATH)

# Composite index settings from the config
HCI_CFG = config["hci_composite"]
# Base weights of the four components
WEIGHTS_BASE = HCI_CFG["weights"]
# Check the base weights add up to 1
assert abs(sum(WEIGHTS_BASE.values()) - 1.0) < 1e-9, "los pesos base del HCI deben sumar 1.0"
MAG_COL_PRIMARY = HCI_CFG["magnitude_column_primary"]      # 'enso_magnitude_mean'
MAG_COL_ALT = HCI_CFG["magnitude_column_alternative"]      # 'enso_magnitude_max'
# Number of classes
N_CLASSES = HCI_CFG["n_classes"]
# Class labels, from lowest to highest
CLASS_LABELS = HCI_CFG["class_labels"]
# Alternative weight scenarios for the sensitivity analysis
WEIGHT_SCENARIOS = HCI_CFG["sensitivity_weight_scenarios"]
# Percentile at which severity is capped for the robustness test
WINSOR_PCTL = HCI_CFG["severity_winsorize_percentile"]
# Log weights, magnitude column and classification settings
logger.info("Pesos base: %s | Magnitud primaria: %s | Clasificación: %s (%d clases)",
            WEIGHTS_BASE, MAG_COL_PRIMARY, HCI_CFG["classification_method"], N_CLASSES)

## 1. Carga y unión de los cuatro índices

Se cargan las cuatro tablas de cantón (Fases 4-7) y se unen por
`province_code`/`province_name_official`/`canton_code`/`canton_name_official`/
`hazard_type` — la misma llave que ya comparten por construcción (todas
derivan de `hazard_event_geo.gpkg` con el mismo join de atributo sobre
`canton_code`). Se repite el mismo proceso a nivel provincia.

**English:** 1. Load and join the four indices

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Keys that identify a canton view
KEYS_CANTON = ["province_code", "province_name_official", "canton_code", "canton_name_official", "hazard_type"]
# Keys that identify a province view
KEYS_PROVINCIA = ["province_code", "province_name_official", "hazard_type"]


# Load and join the four indices for one administrative level
def cargar_y_unir(nivel: str, keys: list[str]) -> pd.DataFrame:
    """Carga y une las 4 tablas de índices (Severidad, Recurrencia,
    Persistencia, Magnitud ENSO) de un nivel administrativo dado.

    Args:
        nivel: 'canton' o 'provincia' — sufijo de archivo.
        keys: columnas de unión.

    Returns:
        DataFrame único con severity_index, recurrence_index,
        persistence_index, enso_magnitude_max, enso_magnitude_mean, y
        n_eventos (validado como consistente entre las 4 tablas de origen).

    English:
    Load the four component indices for one administrative level and join them.

    Args:
        nivel: 'canton' or 'provincia'.
        keys: Columns that identify a unit and hazard view.

    Returns:
        Table with recurrence, severity, persistence and ENSO magnitude per unit.
    """
    # Load the severity index
    sev = pd.read_parquet(processed_dir / f"severity_index_{nivel}.parquet")
    # Load the recurrence index
    rec = pd.read_parquet(processed_dir / f"recurrence_index_{nivel}.parquet")
    # Load the persistence index
    per = pd.read_parquet(processed_dir / f"persistence_index_{nivel}.parquet")
    # Load the ENSO magnitude index
    mag = pd.read_parquet(processed_dir / f"enso_magnitude_index_{nivel}.parquet")

    # Chequeo de consistencia: n_eventos debe coincidir entre Severidad,
    # Persistencia y Magnitud (las 3 que lo calculan directamente sobre
    # hazard_event_geo) — si no coincidiera, señalaría un error de unión
    # en alguna de las fases anteriores.
    # Cross-check the event counts of two indices
    check = sev[keys + ["n_eventos"]].merge(per[keys + ["n_eventos"]], on=keys, suffixes=("_sev", "_per"))
    # Check both indices count the same events
    assert (check["n_eventos_sev"] == check["n_eventos_per"]).all(), \
        "n_eventos no coincide entre Severidad y Persistencia — revisar fases anteriores"

    # Join the four indices on the unit keys
    out = sev[keys + ["n_eventos", "severity_index"]].merge(
        rec[keys + ["recurrence_index"]], on=keys, how="inner"
    ).merge(
        per[keys + ["persistence_index"]], on=keys, how="inner"
    ).merge(
        mag[keys + [MAG_COL_PRIMARY, MAG_COL_ALT]], on=keys, how="inner"
    )
    # Return the per-unit table
    return out


# Join the indices at canton level
hci_canton = cargar_y_unir("canton", KEYS_CANTON)
# Join the indices at province level
hci_provincia = cargar_y_unir("provincia", KEYS_PROVINCIA)
# Log the size of both tables
logger.info("Unión completa — cantón: %d filas | provincia: %d filas", len(hci_canton), len(hci_provincia))

## 2. HCI base (oficial)

`HCI = 0.35·Recurrencia + 0.30·Severidad + 0.20·Persistencia +
0.15·Magnitud ENSO` (Fase 0 §9), usando `enso_magnitude_mean` como insumo
de Magnitud ENSO (Fase 7 §6.3 — mucho mejor poder de discriminación que
`enso_magnitude_max`).

**English:** 2. Base composite index

In [ ]:
# Compute the composite index with a set of weights
def calcular_hci(df: pd.DataFrame, weights: dict[str, float], mag_col: str, out_col: str = "hci") -> pd.DataFrame:
    """Calcula el HCI compuesto para un conjunto de pesos y una columna
    de magnitud ENSO dados.

    Args:
        df: tabla unida con severity_index, recurrence_index,
            persistence_index, y la columna de magnitud indicada.
        weights: dict con claves 'recurrencia','severidad','persistencia','magnitud_enso'.
        mag_col: nombre de la columna de magnitud ENSO a usar.
        out_col: nombre de la columna de salida.

    Returns:
        Copia de df con la columna `out_col` añadida, en [0,1].

    English:
    Compute the composite index as a weighted sum of the four components.

    Args:
        df: Table with the four component columns.
        weights: Weight per component; must add up to 1.
        mag_col: ENSO magnitude column to use.
        out_col: Name of the output column.

    Returns:
        Copy of the table with the composite index in [0, 1].
    """
    # Work on a copy
    df = df.copy()
    # Weighted sum of recurrence, severity, persistence and ENSO magnitude
    df[out_col] = (
        weights["recurrencia"] * df["recurrence_index"]
        + weights["severidad"] * df["severity_index"]
        + weights["persistencia"] * df["persistence_index"]
        + weights["magnitud_enso"] * df[mag_col]
    )
    # Check the index is between 0 and 1
    assert df[out_col].between(0, 1).all(), f"{out_col} fuera de [0,1]"
    # Return the table with the index
    return df


# Base composite index at canton level
hci_canton = calcular_hci(hci_canton, WEIGHTS_BASE, MAG_COL_PRIMARY, "hci")
# Base composite index at province level
hci_provincia = calcular_hci(hci_provincia, WEIGHTS_BASE, MAG_COL_PRIMARY, "hci")
# Log that the base index was computed
logger.info("HCI base calculado (mag=%s) para ambos niveles", MAG_COL_PRIMARY)

## 3. Clasificación por quiebres naturales (Jenks)

Se aplica sobre la vista `"TODOS"` de cada nivel — igual que la
clasificación provisional de Severidad en Fase 4 §4.3, pero esta sí es la
clasificación oficial del proyecto (Fase 0 §7).

**English:** 3. Natural-breaks (Jenks) classification

In [ ]:
# Classify values with Jenks natural breaks
def clasificar_jenks(serie: pd.Series, n_classes: int, labels: list[str]) -> pd.Series:
    """Clasifica una serie continua con quiebres naturales de Jenks.

    Args:
        serie: valores continuos a clasificar (HCI).
        n_classes: número de clases.
        labels: etiquetas ordenadas de menor a mayor.

    Returns:
        Serie categórica con la etiqueta de clase de cada fila.

    English:
    Classify values with Jenks natural breaks.

    Args:
        serie: Values to classify.
        n_classes: Number of classes.
        labels: Class labels, from lowest to highest.

    Returns:
        The class of each value.
    """
    # Compute the class boundaries
    breaks = jenkspy.jenks_breaks(serie.values, n_classes=n_classes)
    # jenks_breaks devuelve n_classes+1 bordes (incluye min y max) — se usa
    # pd.cut con esos bordes exactos; el primer borde se abre ligeramente
    # hacia abajo para que el valor mínimo no quede excluido del primer bin.
    # Convert the boundaries to a list
    breaks = list(breaks)
    # Widen the lowest boundary so the minimum value is included
    breaks[0] -= 1e-9
    # Assign each value to its class
    return pd.cut(serie, bins=breaks, labels=labels, include_lowest=True)


# Select the combined hazard view at canton level
mask_todos_canton = hci_canton["hazard_type"] == "TODOS"
# Classify the canton index
hci_canton.loc[mask_todos_canton, "clase_hci"] = clasificar_jenks(
    hci_canton.loc[mask_todos_canton, "hci"], N_CLASSES, CLASS_LABELS
)

# Select the combined hazard view at province level
mask_todos_prov = hci_provincia["hazard_type"] == "TODOS"
# Classify the province index
hci_provincia.loc[mask_todos_prov, "clase_hci"] = clasificar_jenks(
    hci_provincia.loc[mask_todos_prov, "hci"], N_CLASSES, CLASS_LABELS
)

# Print a header for the class distribution
print("Distribución de clases HCI (cantón, Jenks):")
# Print how many cantons fall in each class
print(hci_canton.loc[mask_todos_canton, "clase_hci"].value_counts().reindex(CLASS_LABELS))

## 4. Análisis de sensibilidad

Cuatro pruebas, cada una dirigida a un parámetro específico ya señalado
como pendiente en una fase anterior. En todas se compara contra el HCI
base (§2) con **correlación de rangos de Spearman** (mide si el *orden*
relativo de los cantones se mantiene, que es lo que realmente importa para
priorizar) y con el **número de cantones que cambian de clase Jenks**
(mide si la sensibilidad tendría consecuencia práctica en la
clasificación, no solo en el valor numérico).

**English:** 4. Sensitivity analysis

In [ ]:
# Compare a base index against an alternative
def resumen_sensibilidad(base: pd.Series, alterna: pd.Series, clase_base: pd.Series, clase_alterna: pd.Series, nombre: str) -> dict:
    """Resume una prueba de sensibilidad: correlación de Spearman entre el
    HCI base y una variante, y cuántas unidades cambian de clase Jenks.

    Args:
        base: HCI base (vista 'TODOS').
        alterna: HCI de la variante a comparar.
        clase_base: clase Jenks del HCI base.
        clase_alterna: clase Jenks recalculada sobre la variante.
        nombre: etiqueta de la prueba, para el reporte impreso.

    Returns:
        Dict con rho de Spearman, p-valor, y conteo/porcentaje de cambios de clase.

    English:
    Compare a base index with an alternative version.

    Args:
        base: Base index values.
        alterna: Alternative index values.
        clase_base: Base classes.
        clase_alterna: Alternative classes.
        nombre: Name of the test.

    Returns:
        Dictionary with the Spearman correlation and the number of units that change class.
    """
    # Spearman correlation between both versions
    rho, p = spearmanr(base, alterna)
    # Number of units that change class
    cambios = (clase_base.values != clase_alterna.values).sum()
    # Dictionary with the comparison results
    resultado = {
        "prueba": nombre, "spearman_rho": round(rho, 4), "p_valor": round(p, 6),
        "cantones_cambian_clase": int(cambios), "pct_cambian_clase": round(100 * cambios / len(base), 1),
    }
    # Print the correlation and the number of class changes
    print(f"{nombre:45s} rho={rho:.4f}  cantones que cambian de clase: {cambios}/{len(base)} ({resultado['pct_cambian_clase']}%)")
    # Return the comparison results
    return resultado


# List that collects all sensitivity tests
resultados_sensibilidad = []

### 4.1 Magnitud ENSO: `enso_magnitude_mean` (base) vs. `enso_magnitude_max` (Fase 7 §6)

**English:** 4.1 ENSO magnitude: mean (base) versus maximum

In [ ]:
# Composite index using the maximum ENSO magnitude
hci_canton = calcular_hci(hci_canton, WEIGHTS_BASE, MAG_COL_ALT, "hci_mag_max")
# Classify the alternative index
hci_canton.loc[mask_todos_canton, "clase_hci_mag_max"] = clasificar_jenks(
    hci_canton.loc[mask_todos_canton, "hci_mag_max"], N_CLASSES, CLASS_LABELS
)
# Compare it against the base index
resultados_sensibilidad.append(resumen_sensibilidad(
    hci_canton.loc[mask_todos_canton, "hci"], hci_canton.loc[mask_todos_canton, "hci_mag_max"],
    hci_canton.loc[mask_todos_canton, "clase_hci"], hci_canton.loc[mask_todos_canton, "clase_hci_mag_max"],
    "Magnitud: mean vs. max",
))

### 4.2 Pesos alternativos (Fase 0 §9 pide explícitamente esta validación)

Tres escenarios además del caso base (`config.yaml`,
`hci_composite.sensitivity_weight_scenarios`): pesos iguales (25% cada
uno), énfasis en Severidad, énfasis en Recurrencia.

**English:** 4.2 Alternative weights

In [ ]:
# Loop over the alternative weight scenarios
for nombre_escenario, pesos in WEIGHT_SCENARIOS.items():
    # Check each scenario's weights add up to 1
    assert abs(sum(pesos.values()) - 1.0) < 1e-9, f"pesos del escenario {nombre_escenario} deben sumar 1.0"
    # Name of the scenario column
    col = f"hci_pesos_{nombre_escenario}"
    # Composite index with the scenario weights
    hci_canton = calcular_hci(hci_canton, pesos, MAG_COL_PRIMARY, col)
    # Name of the scenario class column
    clase_col = f"clase_{col}"
    # Classify the scenario index
    hci_canton.loc[mask_todos_canton, clase_col] = clasificar_jenks(
        hci_canton.loc[mask_todos_canton, col], N_CLASSES, CLASS_LABELS
    )
    # Compare it against the base index
    resultados_sensibilidad.append(resumen_sensibilidad(
        hci_canton.loc[mask_todos_canton, "hci"], hci_canton.loc[mask_todos_canton, col],
        hci_canton.loc[mask_todos_canton, "clase_hci"], hci_canton.loc[mask_todos_canton, clase_col],
        f"Pesos: {nombre_escenario}",
    ))

### 4.3 Robustez de Severidad frente a los 2 valores atípicos (Fase 4 §6)

Se recalcula Severidad con los puntajes de pilar **winsorizados** (recorte
al percentil 99 antes de min-max, en vez de dejar que el valor extremo real
defina el techo) y se rehace el HCI con esa versión — mide cuánto cambia
el resultado final si los dos valores atípicos (Quito, 750,000 afectados
indirectos; Baños de Agua Santa, 500,000 muertes de ganado) fueran errores
de captura en vez de datos válidos.

**Nota metodológica sobre el percentil elegido:** se probó primero con
percentil 95, pero varias columnas de impacto (`affected_indirect`,
`houses_destroyed`, `livestock_deaths`) tienen más del 95% de sus eventos
en 0 — su percentil 95 es literalmente 0. Winsorizar ahí no "recorta un
valor extremo", aplasta casi toda la señal de esa columna a 0, generando un
artefacto de empate masivo (117 de 221 cantones quedaban empatados
exactamente en 0.75 en esa primera prueba) que no mide sensibilidad real a
outliers, sino un umbral mal elegido para columnas tan dispersas. Se
documenta el hallazgo y se usa percentil 99 en su lugar, que preserva señal
real en las 5 columnas (verificado antes de continuar).

**English:** 4.3 Severity robustness to the two outliers

In [ ]:
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Severity index settings from the config
SEV_CFG = config["severity_index"]
# Weight multiplier for destroyed assets versus affected ones
W_DESTROYED = SEV_CFG["destroyed_weight_multiplier"]
# Weight multiplier for deaths and missing persons
W_DEATH = SEV_CFG["death_weight_multiplier"]
# Weight multiplier for injured persons
W_INJURED = SEV_CFG["injured_weight_multiplier"]
# Weights of the five pillars
PILLAR_WEIGHTS = SEV_CFG["pillar_weights"]

# Mismas fórmulas de puntaje por pilar que Fase 4 §4.1 — no se reinventan.
# Population score: affected, evacuated and sheltered people plus weighted deaths, missing and injured
he["score_poblacion"] = (
    he["affected_direct"] + he["affected_indirect"] + he["evacuated"] + he["sheltered"]
    + W_DEATH * (he["deaths"] + he["missing_persons"]) + W_INJURED * he["injured"]
)
# Housing score: affected houses plus weighted destroyed houses
he["score_vivienda"] = he["houses_affected"] + W_DESTROYED * he["houses_destroyed"]
# Agriculture score: affected hectares plus weighted destroyed hectares
he["score_agricultura"] = he["crops_affected_ha"] + W_DESTROYED * he["crops_destroyed_ha"]
# Livestock score: affected animals plus weighted deaths
he["score_ganaderia"] = he["livestock_affected"] + W_DESTROYED * he["livestock_deaths"]
# Infrastructure score: affected and destroyed schools and health facilities
he["score_infraestructura"] = (
    he["schools_affected"] + W_DESTROYED * he["schools_destroyed"]
    + he["health_facilities_affected"] + W_DESTROYED * he["health_facilities_destroyed"]
)
# Names of the five pillar score columns
PILLAR_SCORE_COLS = ["score_poblacion", "score_vivienda", "score_agricultura", "score_ganaderia", "score_infraestructura"]

# Winsorización: cualquier valor por encima del percentil 95 (nacional, por
# columna) se recorta a ese percentil, ANTES de agregar por cantón — así
# ningún evento individual extremo puede, por sí solo, definir el techo
# nacional de min-max.
# Copy the event table for the robustness test
he_wins = he.copy()
# Loop over the pillar scores
for col in PILLAR_SCORE_COLS:
    # Cap value: the chosen percentile of the pillar score
    techo = he_wins[col].quantile(WINSOR_PCTL)
    # Cap extreme events at that value
    he_wins[col] = he_wins[col].clip(upper=techo)

# Load the official canton and province names
adm2 = pyogrio.read_dataframe(processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False)[
    ["canton_code", "canton_name_official", "province_name_official"]
].drop_duplicates()
# Attach official names to the events
he_wins = he_wins.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")

# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]


# Min-max scaling of one series
def _minmax(s: pd.Series) -> pd.Series:
    # Range of the series
    rango = s.max() - s.min()
    # Min-max scaling of one series
    return pd.Series(0.0, index=s.index) if rango == 0 else (s - s.min()) / rango


# Maximum of each pillar per canton using the capped scores
sev_wins = he_wins.groupby(CANTON_COLS)[PILLAR_SCORE_COLS].max().reset_index()
# Loop over the pillar scores
for col in PILLAR_SCORE_COLS:
    # Normalize each pillar
    sev_wins[col.replace("score_", "norm_")] = _minmax(sev_wins[col])
# Severity index computed from the capped scores
sev_wins["severity_index_robusto"] = (
    PILLAR_WEIGHTS["poblacion"] * sev_wins["norm_poblacion"]
    + PILLAR_WEIGHTS["vivienda"] * sev_wins["norm_vivienda"]
    + PILLAR_WEIGHTS["agricultura"] * sev_wins["norm_agricultura"]
    + PILLAR_WEIGHTS["ganaderia"] * sev_wins["norm_ganaderia"]
    + PILLAR_WEIGHTS["infraestructura"] * sev_wins["norm_infraestructura"]
)
# Log how many cantons have the robust severity
logger.info("Severidad robusta (winsorizada al percentil %.0f) calculada para %d cantones", WINSOR_PCTL * 100, len(sev_wins))

In [ ]:
# Attach the robust severity to the composite table
hci_canton = hci_canton.merge(
    sev_wins[CANTON_COLS + ["severity_index_robusto"]], on=CANTON_COLS, how="left"
)
# Composite index using the robust severity
hci_canton["hci_sev_robusta"] = (
    WEIGHTS_BASE["recurrencia"] * hci_canton["recurrence_index"]
    + WEIGHTS_BASE["severidad"] * hci_canton["severity_index_robusto"]
    + WEIGHTS_BASE["persistencia"] * hci_canton["persistence_index"]
    + WEIGHTS_BASE["magnitud_enso"] * hci_canton[MAG_COL_PRIMARY]
)
# Classify the robust version
hci_canton.loc[mask_todos_canton, "clase_hci_sev_robusta"] = clasificar_jenks(
    hci_canton.loc[mask_todos_canton, "hci_sev_robusta"], N_CLASSES, CLASS_LABELS
)
# Compare it against the base index
resultados_sensibilidad.append(resumen_sensibilidad(
    hci_canton.loc[mask_todos_canton, "hci"], hci_canton.loc[mask_todos_canton, "hci_sev_robusta"],
    hci_canton.loc[mask_todos_canton, "clase_hci"], hci_canton.loc[mask_todos_canton, "clase_hci_sev_robusta"],
    "Severidad: original vs. winsorizada (outliers)",
))

# Caso específico: ¿cómo cambia Quito y Chone (los 2 casos más comentados en Fase 4) con la versión robusta?
# Select the three cantons affected by the extreme events
casos = hci_canton.loc[mask_todos_canton & hci_canton.canton_name_official.isin(
    ["Distrito Metropolitano de Quito", "Chone", "Baños de Agua Santa"]
)]
# Print the original and robust values for those cantons
print(casos[["canton_name_official", "severity_index", "severity_index_robusto", "hci", "hci_sev_robusta"]].to_string(index=False))

### 4.4 Resumen consolidado de sensibilidad

**English:** 4.4 Consolidated sensitivity summary

In [ ]:
# Collect all sensitivity results in one table
tabla_sensibilidad = pd.DataFrame(resultados_sensibilidad)
# Print the sensitivity summary
print(tabla_sensibilidad.to_string(index=False))
# Save the sensitivity summary
tabla_sensibilidad.to_csv(processed_dir / "hci_sensitivity_summary.csv", index=False)
# Log the number of tests
logger.info("Resumen de sensibilidad exportado: %d pruebas", len(tabla_sensibilidad))

## 5. Validación

**English:** 5. Validation

In [ ]:
# Expected number of cantons
n_cantones_esperados = 221
# Number of cantons in the classified view
n_en_clase = mask_todos_canton.sum()
# Check every canton is classified
assert n_en_clase == n_cantones_esperados, "cobertura incompleta en la clasificación HCI (cantón)"
# Check every province is classified
assert hci_provincia.loc[mask_todos_prov].shape[0] == 24, "cobertura incompleta en la clasificación HCI (provincia)"
# Check the composite index is between 0 and 1
assert hci_canton["hci"].between(0, 1).all()
# Check every canton has a class
assert hci_canton["clase_hci"].notna().sum() == n_cantones_esperados == (~hci_canton.loc[mask_todos_canton, "clase_hci"].isna()).sum()
# Log that coverage and range are valid
logger.info("Validación de cobertura y rango del HCI: OK (221 cantones, 24 provincias, todo en [0,1])")

## 6. Exportación

**English:** 6. Export

In [ ]:
# Tables to export
tablas = {
    "hci_composite_canton": hci_canton,
    "hci_composite_provincia": hci_provincia,
}
# Loop over the tables
for nombre, df in tablas.items():
    # Save as Parquet
    df.to_parquet(processed_dir / f"{nombre}.parquet", index=False)
    # Save as CSV
    df.to_csv(processed_dir / f"{nombre}.csv", index=False)
    # Log each exported table
    logger.info("Exportado: %s (%d filas)", nombre, len(df))

# Print the export folder
print(f"Tablas exportadas a {processed_dir} :")
# Loop over the exported tables
for nombre in tablas:
    # Print each table name
    print(f" - {nombre}")
# Print the sensitivity summary table name
print(" - hci_sensitivity_summary")

## 7. Resultados — ranking nacional del HCI

**English:** 7. Results: national ranking of the index

In [ ]:
# Fifteen cantons with the highest composite index
top15_canton = hci_canton.loc[mask_todos_canton].sort_values("hci", ascending=False).head(15)
# Print a header for the canton ranking
print("Top 15 cantones por HCI:")
# Print the canton ranking
print(top15_canton[["province_name_official", "canton_name_official", "hci", "clase_hci",
                     "severity_index", "recurrence_index", "persistence_index", MAG_COL_PRIMARY]].to_string(index=False))

# Print an empty line
print()
# Ten provinces with the highest composite index
top10_prov = hci_provincia.loc[mask_todos_prov].sort_values("hci", ascending=False).head(10)
# Print a header for the province ranking
print("Top 10 provincias por HCI:")
# Print the province ranking
print(top10_prov[["province_name_official", "hci", "clase_hci"]].to_string(index=False))

## 8. Próximos pasos

- **Fase 9 — Análisis espacial**: hotspots (Getis-Ord Gi*, Moran's I local),
  densidad de kernel, agregación hexagonal (Fase 0 §8).
- **Fase 10 — Productos**: mapas de cronicidad, perfiles provinciales y
  cantonales, tablero interactivo, reportes automatizados (Fase 0 §9).
- Validación externa contra fuentes independientes (UNOSAT, EM-DAT,
  DesInventar, Copernicus EMS — Fase 0 §12), pendiente en todas las fases
  anteriores.
- Revisar con Jerson el resumen de sensibilidad de la sección 4.4: si algún
  escenario cambia sustancialmente el ranking o la clasificación de muchos
  cantones, decidir si el HCI base se ajusta antes de considerarse
  definitivo para uso operacional en Acción Anticipatoria.